In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from tqdm.notebook import tqdm

# --- 1. Setup and Data Loading ---

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
tqdm.pandas()

# Load ALOFT dataset with metadata
file_path = '../data/processed/public/ALOFT_with_metadata.csv'
try:
    df = pd.read_csv(file_path)
    print("ALOFT Dataset loaded successfully.")
    print(f"Total rows in dataset: {len(df)}")
    
    # Display basic information about the dataset
    print("\n" + "="*60)
    print("DATASET OVERVIEW")
    print("="*60)
    
    # Column information
    print(f"\nColumns ({len(df.columns)}):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\nDataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Data types
    print(f"\nData Types:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {dtype}: {count} columns")
    
    # Missing values summary
    print(f"\nMissing Values:")
    missing_data = df.isnull().sum()
    missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
    if len(missing_data) > 0:
        print(f"  Columns with missing values: {len(missing_data)}")
        for col, missing_count in missing_data.items():
            missing_pct = (missing_count / len(df)) * 100
            print(f"    {col}: {missing_count:,} ({missing_pct:.1f}%)")
    else:
        print("  No missing values found")
    
    # Text columns analysis
    text_columns = ['Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'T50 Quote', 
                   'Non-Literary Baseline', 'Goodreads Popular Quote', 'Google Books Page Text', 'T50 Quote-Free Context', 'T50 Quote-Free Context Length Matched']
    
    print(f"\nText Columns Analysis:")
    for col in text_columns:
        if col in df.columns:
            # Calculate text length statistics
            text_lengths = df[col].astype(str).str.len()
            non_null_count = df[col].notna().sum()
            # Calculate unique authors and books
            unique_authors = df['Author of Goodreads Sample Book'].nunique()
            unique_books = df['Title of Goodreads Sample Book'].nunique()

            print("--- Basic Statistics ---")
            print(f"Number of unique authors: {unique_authors}")
            print(f"Number of unique books: {unique_books}")
            print(f"  {col}:")
            print(f"    Non-null entries: {non_null_count:,}")
            if non_null_count > 0:
                print(f"    Avg length: {text_lengths.mean():.1f} characters")
                print(f"    Length range: {text_lengths.min()}-{text_lengths.max()} characters")
    
    # Metadata columns analysis
    metadata_columns = ['AUTHOR', 'TITLE', 'language_or_country', 'publication_date', 'genres']
    
    print(f"\nMetadata Analysis:")
    for col in metadata_columns:
        if col in df.columns:
            unique_count = df[col].nunique()
            non_null_count = df[col].notna().sum()
            print(f"  {col}:")
            print(f"    Unique values: {unique_count:,}")
            print(f"    Non-null entries: {non_null_count:,}")
            
            # Show top values for categorical columns
            if col in ['AUTHOR', 'TITLE', 'language_or_country', 'genres']:
                top_values = df[col].value_counts().head(3)
                print(f"    Top 3 values:")
                for value, count in top_values.items():
                    print(f"      '{value}': {count}")
    
    # Numerical column analysis (LIKES)
    if 'LIKES' in df.columns:
        print(f"\nLikes Analysis:")
        likes_stats = df['LIKES'].describe()
        print(f"  Count: {likes_stats['count']:,.0f}")
        print(f"  Mean: {likes_stats['mean']:.1f}")
        print(f"  Median: {likes_stats['50%']:.1f}")
        print(f"  Min-Max: {likes_stats['min']:.0f}-{likes_stats['max']:.0f}")
        print(f"  Standard deviation: {likes_stats['std']:.1f}")
    
    print("\n" + "="*60)
    print("First 3 rows preview:")
    print("="*60)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 100)
    print(df.head(3))
    
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}")
    df = pd.DataFrame()
except Exception as e:
    print(f"Error loading dataset: {e}")
    df = pd.DataFrame()


In [ ]:
# ==============================================================================
# MASTER SETUP CELL
# ==============================================================================
# This cell should be run once at the beginning of the notebook. It defines
# ALL global configuration, helper classes, and color palettes needed
# for the subsequent analysis cells.

import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------- 1. Results Management -----------------------------
class ResultsManager:
    """
    A class to manage saving plots, tables, and text examples for the thesis.
    Ensures all outputs are stored in a structured directory hierarchy.
    """
    def __init__(self, base_path='Analysis/Outputs'):
        self.base_path = base_path
        if not os.path.exists(self.base_path):
            os.makedirs(self.base_path)
        print(f"ResultsManager initialized. Outputs will be saved in: {self.base_path}")

    def _get_metric_path(self, metric_name):
        metric_path = os.path.join(self.base_path, metric_name)
        if not os.path.exists(metric_path):
            os.makedirs(metric_path)
        return metric_path

    def save_plot(self, fig, metric_name, filename):
        metric_path = self._get_metric_path(metric_name)
        save_path = os.path.join(metric_path, filename)
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Plot saved to: {save_path}")

    def save_table(self, dataframe, metric_name, filename):
        metric_path = self._get_metric_path(metric_name)
        save_path = os.path.join(metric_path, filename)
        dataframe.to_csv(save_path)
        print(f"✓ Table saved to: {save_path}")

    def save_master_df(self, dataframe, filename='ALOFT_master_metrics.csv'):
        save_path = os.path.join(self.base_path, filename)
        dataframe.to_csv(save_path, index=False)
        print(f"Master metrics DataFrame saved to: {save_path}")

    def save_text_examples(self, highest_df, lowest_df, metric_name, filename, text_col, metric_col, metric_unit, low_label, high_label):
        metric_path = self._get_metric_path(metric_name)
        save_path = os.path.join(metric_path, filename)
        output_str = f"Analysis of: {text_col}\nMetric: {metric_unit}\n" + "="*80 + "\n\n"
        output_str += f"LOWEST {len(lowest_df)} SCORES ({low_label})\n" + "-"*80 + "\n"
        for _, row in lowest_df.iterrows():
            output_str += f"\n[{metric_unit}: {row[metric_col]:.2f}]\n\"{row[text_col]}\"\n" + "-"*25 + "\n"
        output_str += "\n\n" + "="*80 + "\n\n"
        output_str += f"HIGHEST {len(highest_df)} SCORES ({high_label})\n" + "-"*80 + "\n"
        for _, row in highest_df.iloc[::-1].iterrows():
            output_str += f"\n[{metric_unit}: {row[metric_col]:.2f}]\n\"{row[text_col]}\"\n" + "-"*25 + "\n"
        with open(save_path, 'w', encoding='utf-8') as f: f.write(output_str)
        print(f"✓ Text examples saved to: {save_path}")

# --- Instantiate the manager ---
manager = ResultsManager()

# ---------------------------- 2. Color Palette --------------------------------
# For Linguistic Analysis Plots
COLOR_SAMPLE_QUOTE = '#1F77B4'
COLOR_POPULAR_QUOTE = '#17BECF'
COLOR_T50_QUOTE = '#8E3A9B'
COLOR_T50_QUOTE_FREE = '#FF7F0E'
COLOR_T50_QUOTE_FREE_LENGTH_MATCHED = '#E377C2'
COLOR_MATCHED_SNIPPET = '#2CA02C'
COLOR_PAGE_TEXT = '#BCBD22'
COLOR_NONLIT_BASELINE = '#D62728'

# For EDA / Metadata Plots
COLOR_METADATA = '#6B7280'

# --- Master Palette Dictionary (for linguistic analysis cells) ---
palette = {
    "Goodreads Sample Quote": COLOR_SAMPLE_QUOTE,
    "Goodreads Popular Quote": COLOR_POPULAR_QUOTE,
    "Google Books Length Matched Snippet": COLOR_MATCHED_SNIPPET,
    "Google Books Page Text": COLOR_PAGE_TEXT,
    "T50 Quote": COLOR_T50_QUOTE,
    "T50 Quote-Free Context": COLOR_T50_QUOTE_FREE,
    "T50 Quote-Free Context Length Matched": COLOR_T50_QUOTE_FREE_LENGTH_MATCHED,
    "Non-Literary Baseline": COLOR_NONLIT_BASELINE,
}

# --- Comprehensive Color Dictionary (for EDA cells) ---
THESIS_COLORS = {
    'Goodreads Sample Quote': COLOR_SAMPLE_QUOTE,
    'Likes of Sample Quote': COLOR_SAMPLE_QUOTE,
    'Goodreads Popular Quote': COLOR_POPULAR_QUOTE,
    'Likes of Popular Quote': COLOR_POPULAR_QUOTE,
    'T50 Quote': COLOR_T50_QUOTE,
    'T50 Quote-Free Context': COLOR_T50_QUOTE_FREE,
    'T50 Quote-Free Context Length Matched': COLOR_T50_QUOTE_FREE_LENGTH_MATCHED,
    'Google Books Length Matched Snippet': COLOR_MATCHED_SNIPPET,
    'Google Books Page Text': COLOR_PAGE_TEXT,
    'Non-Literary Baseline': COLOR_NONLIT_BASELINE,
    'Author of Goodreads Sample Book': COLOR_METADATA,
    'Title of Goodreads Sample Book': COLOR_METADATA,
    'Language of Author of Goodreads Sample Book': COLOR_METADATA,
    'Publication Date of Goodreads Sample Book': COLOR_METADATA,
    'Genre of Goodreads Sample Book': COLOR_METADATA
}
print("🎨 Color palettes defined and ready.")

# -------------------------- 3. Source Mappings --------------------------------
sources_readability = { "Goodreads Sample Quote": "sample", "Goodreads Popular Quote": "popular", "Google Books Length Matched Snippet": "snippet", "Google Books Page Text": "page", "T50 Quote": "t50", "T50 Quote-Free Context": "t50free", "T50 Quote-Free Context Length Matched": "t50freelength", "Non-Literary Baseline": "nonlit" }
sources_lex_div = { "Goodreads Sample Quote": "sample_lex_div", "Goodreads Popular Quote": "popular_lex_div", "Google Books Length Matched Snippet": "matched_snippet_lex_div", "Google Books Page Text": "page_text_lex_div", "T50 Quote": "t50_quote_lex_div", "T50 Quote-Free Context": "t50_quote_free_context_lex_div", "T50 Quote-Free Context Length Matched": "t50_quote_free_context_length_matched_lex_div", "Non-Literary Baseline": "nonlit_baseline_lex_div" }
sources_entropy = { "Goodreads Sample Quote": "sample_entropy", "Goodreads Popular Quote": "popular_entropy", "Google Books Length Matched Snippet": "snippet_entropy", "Google Books Page Text": "page_entropy", "T50 Quote": "t50_entropy", "T50 Quote-Free Context": "t50free_entropy", "T50 Quote-Free Context Length Matched": "t50freelength_entropy", "Non-Literary Baseline": "nonlit_entropy" }
print("Source mappings defined and ready.")

# ------------------------ 4. Shared Helper Functions --------------------------
def cliffs_delta(a, b):
    m, n = len(a), len(b);
    if m == 0 or n == 0: return np.nan
    gt = sum(x > y for x in a for y in b)
    lt = sum(x < y for x in a for y in b)
    return (gt - lt) / (m * n)

print("Shared helper functions defined and ready.")
print("\nMaster Setup Cell complete. All configurations are now loaded globally.")

In [ ]:
# ==============================================================================
# Cell 2: Color Palette Visualization
# ==============================================================================
# This cell displays the color swatches for the thesis palette.
# It uses the color variables and the 'palette' dictionary defined globally
# in the Master Setup Cell.

print("🎨 Displaying Thesis Color Palette Swatches...")

# Create visual rectangles to preview each color
fig, axes = plt.subplots(4, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [1, 1, 1, 0.5]})
fig.suptitle('Thesis Color Palette Reference', fontsize=16, fontweight='bold')

# Goodreads Colors
goodreads_labels = {
    'Goodreads Sample Quote': COLOR_SAMPLE_QUOTE,
    'Goodreads Popular Quote': COLOR_POPULAR_QUOTE
}
for i, (label, color) in enumerate(goodreads_labels.items()):
    axes[0].add_patch(plt.Rectangle((i, 0), 0.95, 1, facecolor=color, edgecolor='black'))
    axes[0].text(i + 0.475, 0.5, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
axes[0].set_xlim(0, len(goodreads_labels)); axes[0].set_ylim(0, 1)
axes[0].set_title('Goodreads Corpora (Shades of Blue/Cyan)'); axes[0].axis('off')

# T50 Colors
t50_labels = {
    'T50 Quote': COLOR_T50_QUOTE,
    'T50 Quote-Free Context': COLOR_T50_QUOTE_FREE,
    'T50 Quote-Free Context Length Matched': COLOR_T50_QUOTE_FREE_LENGTH_MATCHED
}
for i, (label, color) in enumerate(t50_labels.items()):
    axes[1].add_patch(plt.Rectangle((i, 0), 0.95, 1, facecolor=color, edgecolor='black'))
    axes[1].text(i + 0.475, 0.5, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
axes[1].set_xlim(0, len(t50_labels)); axes[1].set_ylim(0, 1)
axes[1].set_title('T50 Corpora (Shades of Purple/Orange/Pink)'); axes[1].axis('off')

# Google Books & Baseline Colors
other_labels = {
    'Google Books Length Matched Snippet': COLOR_MATCHED_SNIPPET,
    'Google Books Page Text': COLOR_PAGE_TEXT,
    'Non-Literary Baseline': COLOR_NONLIT_BASELINE
}
for i, (label, color) in enumerate(other_labels.items()):
    axes[2].add_patch(plt.Rectangle((i, 0), 0.95, 1, facecolor=color, edgecolor='black'))
    axes[2].text(i + 0.475, 0.5, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
axes[2].set_xlim(0, len(other_labels)); axes[2].set_ylim(0, 1)
axes[2].set_title('Google Books & Baseline Corpora (Shades of Green/Red)'); axes[2].axis('off')

# Clear empty subplot
axes[3].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# --- Modern Quote Likes Distribution Visualization ---

# Ensure both like columns are numeric
df['Likes of Sample Quote'] = pd.to_numeric(df['Likes of Sample Quote'], errors='coerce').fillna(0)
df['Likes of Popular Quote'] = pd.to_numeric(df['Likes of Popular Quote'], errors='coerce').fillna(0)

# Filter out zero likes for better visualization
sample_likes = df[df['Likes of Sample Quote'] > 0]['Likes of Sample Quote']
popular_likes = df[df['Likes of Popular Quote'] > 0]['Likes of Popular Quote']

# Modern, clean matplotlib styling
plt.style.use('default')  # Reset any previous styles
fig, ax = plt.subplots(figsize=(12, 7))

# Smart color approach - use contrasting colors that work well when overlapping
sample_color = THESIS_COLORS['Likes of Sample Quote']
popular_color = THESIS_COLORS['Likes of Popular Quote'] 

# Create histograms with shared bins for proper overlay
# Determine the range for both datasets
min_likes = min(sample_likes.min(), popular_likes.min())
max_likes = max(sample_likes.max(), popular_likes.max())

# Create shared bins with EQUAL WIDTH (linear spacing)
n_bins = 100
bins = np.linspace(min_likes, max_likes, n_bins)

# Create overlaid histograms with smart transparency and edge styling
ax.hist(sample_likes, bins=bins, alpha=0.8, color=sample_color, 
        label='Goodreads Sample', edgecolor='white', linewidth=0.5,
        histtype='stepfilled')

ax.hist(popular_likes, bins=bins, alpha=0.6, color=popular_color,
        label='Goodreads Popular', edgecolor='white', linewidth=0.5,
        histtype='stepfilled')

# Professional styling - clean and minimal
ax.set_title('Distribution of Quote Likes', fontsize=18, fontweight='bold', 
             fontfamily='Arial', pad=20)
ax.set_xlabel('Number of Likes', fontsize=14, fontfamily='Arial')
ax.set_ylabel('Number of Quotes', fontsize=14, fontfamily='Arial')

# Log scale for better visualization
ax.set_yscale('log')

# Remove grid lines for clean look
ax.grid(False)

# Clean axes styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Professional legend
ax.legend(frameon=True, fancybox=True, shadow=False, 
          fontsize=12, loc='upper right', framealpha=0.9)

# Clean tick styling
ax.tick_params(axis='both', which='major', labelsize=11, colors='#333333')
ax.tick_params(axis='both', which='minor', labelsize=9, colors='#666666')

# Tight layout for better spacing
plt.tight_layout()
plt.show()

# Summary statistics with professional formatting
print("\n" + "="*60)
print("QUOTE LIKES ANALYSIS")
print("="*60)

print(f"\nSample Quote Likes:")
print(f"  • Mean: {sample_likes.mean():.1f} likes")
print(f"  • Median: {sample_likes.median():.1f} likes") 
print(f"  • Maximum: {sample_likes.max():,} likes")
print(f"  • Quotes with likes: {len(sample_likes):,} ({len(sample_likes)/len(df)*100:.1f}%)")

print(f"\nPopular Quote Likes:")
print(f"  • Mean: {popular_likes.mean():.1f} likes")
print(f"  • Median: {popular_likes.median():.1f} likes")
print(f"  • Maximum: {popular_likes.max():,} likes") 
print(f"  • Quotes with likes: {len(popular_likes):,} ({len(popular_likes)/len(df)*100:.1f}%)")

print(f"\nComparative Analysis:")
print(f"  • Popular quotes receive {popular_likes.mean()/sample_likes.mean():.1f}x more likes on average")
print(f"  • Median difference: {popular_likes.median() - sample_likes.median():.0f} likes")
print("="*60)

In [ ]:
# --- Modern Publication Date Distribution ---

# Ensure 'Publication Date of Goodreads Sample Book' is a numeric type, coercing errors
df['Publication Date of Goodreads Sample Book'] = pd.to_numeric(df['Publication Date of Goodreads Sample Book'], errors='coerce')
df_dates = df.dropna(subset=['Publication Date of Goodreads Sample Book']).copy()

# Filter out unrealistic dates (e.g., year 0 or future dates)
current_year = pd.to_datetime('today').year
df_dates = df_dates[df_dates['Publication Date of Goodreads Sample Book'].between(1, current_year)]

# --- Modern Plotting with Clean Styling ---
start_year_for_plot = 1550
end_year_for_plot = df_dates['Publication Date of Goodreads Sample Book'].max()

# Find the outliers published before our start year
outliers = df_dates[df_dates['Publication Date of Goodreads Sample Book'] < start_year_for_plot]
unique_outliers = outliers.sort_values('Publication Date of Goodreads Sample Book').drop_duplicates(subset=['Title of Goodreads Sample Book', 'Author of Goodreads Sample Book'])

# Create modern, clean visualization
plt.style.use('default')  # Reset any previous styles
fig, ax = plt.subplots(figsize=(14, 8))

# Create histogram with modern styling using metadata color
publication_years = df_dates['Publication Date of Goodreads Sample Book']
n, bins, patches = ax.hist(publication_years, bins=75, alpha=0.8, 
                          color=COLOR_METADATA, edgecolor='white', linewidth=0.5)

# Professional styling
ax.set_title('Distribution of Book Publication Dates', fontsize=18, fontweight='bold', 
             fontfamily='Arial', pad=20)
ax.set_xlabel('Year of Publication', fontsize=14, fontfamily='Arial')
ax.set_ylabel('Number of Books', fontsize=14, fontfamily='Arial')

# Set the x-axis limits to zoom in on the relevant period
ax.set_xlim(start_year_for_plot, end_year_for_plot + 5)

# Clean axes styling - remove grid and background
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Clean tick styling
ax.tick_params(axis='both', which='major', labelsize=11, colors='#333333')
ax.tick_params(axis='both', which='minor', labelsize=9, colors='#666666')

# Add a clean density curve overlay
from scipy import stats
years_clean = publication_years[(publication_years >= start_year_for_plot) & (publication_years <= end_year_for_plot)]
if len(years_clean) > 10:  # Only add KDE if we have enough data points
    kde = stats.gaussian_kde(years_clean)
    x_range = np.linspace(start_year_for_plot, end_year_for_plot, 300)
    kde_values = kde(x_range) * len(years_clean) * (bins[1] - bins[0])  # Scale to match histogram
    ax.plot(x_range, kde_values, color='#1A237E', linewidth=3, alpha=0.9)

# --- Modern Info Display for Pre-1550 Books ---
if not unique_outliers.empty:
    # Header with improved styling
    header_text = "Pre-1550 Books"
    
    ax.text(0.03, 0.97, header_text, transform=ax.transAxes, 
            fontsize=14, fontweight='bold', ha='left', va='top', 
            color='#1F2937', fontfamily='Arial')
    
    # Book entries with improved formatting and spacing
    y_offset = 0.92  # Increased space after title
    for idx, row in unique_outliers.iterrows():
        year = int(row['Publication Date of Goodreads Sample Book'])
        title = row['Title of Goodreads Sample Book']
        author = row['Author of Goodreads Sample Book']
        
        # Year in bold, followed by title and author
        year_text = f"• {year}"
        book_info = f" {title}, {author}"
        
        # Plot year in bold
        ax.text(0.03, y_offset, year_text, transform=ax.transAxes, 
                fontsize=11, fontweight='bold', ha='left', va='top', 
                color='#4B5563', fontfamily='Arial')
        
        # Plot title and author in regular weight (reduced spacing)
        ax.text(0.065, y_offset, book_info, transform=ax.transAxes, 
                fontsize=11, ha='left', va='top', 
                color='#4B5563', fontfamily='Arial')
        
        y_offset -= 0.035  # Spacing between entries

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("PUBLICATION DATE ANALYSIS")
print("="*60)
print(f"Total books with publication dates: {len(df_dates):,}")
print(f"Date range: {int(df_dates['Publication Date of Goodreads Sample Book'].min())} - {int(df_dates['Publication Date of Goodreads Sample Book'].max())}")
print(f"Median publication year: {int(df_dates['Publication Date of Goodreads Sample Book'].median())}")
print(f"Most common decade: {int(df_dates['Publication Date of Goodreads Sample Book'].mode().iloc[0] // 10 * 10)}s")
print(f"Books before 1550: {len(unique_outliers)}")
print("="*60)

In [ ]:
# --- Modern Author Language Distribution ---

# Normalize language data
language_map = {
    'English': 'English',
    'United States': 'English',
    'United Kingdom': 'English',
    'Canada': 'English',
    'Australia': 'English',
    'Ireland': 'English',
    'American': 'English',
    'British': 'English',
    'UK': 'English',
    'USA': 'English',
    'US': 'English',
    'Britain': 'English',
    'Great Britain': 'English',
    'New Zealand': 'English',
    'South Africa': 'English',
    'British English': 'English',
    'American English': 'English',
    'English (US)': 'English',
    'English (UK)': 'English',
    'English (Canada)': 'English',
    'English (Australia)': 'English',
    'English (New Zealand)': 'English',
    'English (South Africa)': 'English',
    'Early Modern English': 'English',
    'Australian English': 'English',
    'Poland': 'Polish',
    'France': 'French',
    'Germany': 'German',
    'Italy': 'Italian',
    'Spain': 'Spanish',
    'Portugal': 'Portuguese',
    'Brazil': 'Portuguese',
    'Argentina': 'Spanish',
    'Switzerland': 'Swiss',
    'India': 'Hindi',
    'China': 'Chinese',
    'Japan': 'Japanese',
    'Korea': 'Korean',
    'Vietnam': 'Vietnamese',
    'Thailand': 'Thai',
    'Indonesia': 'Indonesian',
    'Austria': 'German',
    'Netherlands': 'Dutch',
    'Belgium': 'Dutch',
    'Sweden': 'Swedish',
    'Norway': 'Norwegian',
    'Denmark': 'Danish',
    'Finland': 'Finnish',
}

# Use .get(x, x) to map known languages, and keep others as they are
df['language_normalized'] = df['Language of Author of Goodreads Sample Book'].apply(lambda x: language_map.get(x, x))

# Get language counts and order by frequency (biggest to smallest)
language_counts = df['language_normalized'].value_counts()
top_languages = language_counts.head(15)  # Show top 15 languages

# Reverse order for horizontal bar chart (biggest at top)
top_languages_reversed = top_languages[::-1]

# Modern, clean visualization
plt.style.use('default')
fig, ax = plt.subplots(figsize=(12, 8))

# Create horizontal bar chart with modern styling
bars = ax.barh(range(len(top_languages_reversed)), top_languages_reversed.values, 
               color=COLOR_METADATA, alpha=0.8, edgecolor='white', linewidth=0.8)

# Professional styling
ax.set_title('Distribution of Author Languages', fontsize=18, fontweight='bold', 
             fontfamily='Arial', pad=20)
ax.set_xlabel('Number of Quotes', fontsize=14, fontfamily='Arial')
ax.set_ylabel('Language / Country', fontsize=14, fontfamily='Arial')

# Set y-axis labels
ax.set_yticks(range(len(top_languages_reversed)))
ax.set_yticklabels(top_languages_reversed.index, fontsize=11)

# Clean axes styling
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Add value labels on bars
max_value = max(top_languages_reversed.values)
for i, (language, count) in enumerate(top_languages_reversed.items()):
    ax.text(count + max_value * 0.01, i, f'{count:,}', 
            va='center', fontsize=10, fontweight='bold')

# Clean tick styling
ax.tick_params(axis='both', which='major', labelsize=11, colors='#333333')
ax.tick_params(axis='x', which='minor', labelsize=9, colors='#666666')

plt.tight_layout()
plt.show()

# Summary statistics
print("\n" + "="*60)
print("AUTHOR LANGUAGE ANALYSIS")
print("="*60)
print(f"Total unique languages: {df['language_normalized'].nunique()}")
print(f"Most common language: {language_counts.index[0]} ({language_counts.iloc[0]:,} quotes)")
print(f"English-speaking percentage: {(language_counts.get('English', 0) / len(df) * 100):.1f}%")
print("="*60)

In [ ]:
# --- Modern Genre Distribution ---

# Get genre counts and select top genres (biggest to smallest)
genre_counts = df['Genre of Goodreads Sample Book'].dropna().value_counts()
top_genres = genre_counts.nlargest(15)  # Show top 15 genres for better readability

# Reverse order for horizontal bar chart (biggest at top)
top_genres_reversed = top_genres[::-1]

# Modern, clean visualization  
plt.style.use('default')
fig, ax = plt.subplots(figsize=(12, 10))

# Create horizontal bar chart with modern styling
bars = ax.barh(range(len(top_genres_reversed)), top_genres_reversed.values,
               color=COLOR_METADATA, alpha=0.8, edgecolor='white', linewidth=0.8)

# Professional styling
ax.set_title('Top 15 Most Common Genres', fontsize=18, fontweight='bold',
             fontfamily='Arial', pad=20)
ax.set_xlabel('Number of Quotes', fontsize=14, fontfamily='Arial')
ax.set_ylabel('Genre', fontsize=14, fontfamily='Arial')

# Set y-axis labels
ax.set_yticks(range(len(top_genres_reversed)))
ax.set_yticklabels(top_genres_reversed.index, fontsize=11)

# Clean axes styling
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Add value labels on bars
max_value = max(top_genres_reversed.values)
for i, (genre, count) in enumerate(top_genres_reversed.items()):
    ax.text(count + max_value * 0.01, i, f'{count:,}',
            va='center', fontsize=10, fontweight='bold')

# Clean tick styling
ax.tick_params(axis='both', which='major', labelsize=11, colors='#333333')
ax.tick_params(axis='x', which='minor', labelsize=9, colors='#666666')

plt.tight_layout()
plt.show()

# Summary statistics
print("\n" + "="*60)
print("GENRE ANALYSIS")
print("="*60)
print(f"Total unique genres: {df['Genre of Goodreads Sample Book'].nunique()}")
print(f"Most common genre: {genre_counts.index[0]} ({genre_counts.iloc[0]:,} quotes)")
print(f"Top 3 genres account for: {(genre_counts.head(3).sum() / genre_counts.sum() * 100):.1f}% of all quotes")

# Show genre diversity
total_quotes_with_genre = genre_counts.sum()
print(f"Quotes with genre data: {total_quotes_with_genre:,} out of {len(df):,} total")
print("="*60)

In [ ]:
# Calculate word counts for each text column
df['quote_length'] = df['Goodreads Sample Quote'].str.split().str.len()
df['ocr_page_length'] = df['Google Books Page Text'].str.split().str.len()
df['matched_snippet_length'] = df['Google Books Length Matched Snippet'].str.split().str.len()

# Modern, clean visualization with thesis color scheme
plt.style.use('default')
plt.figure(figsize=(15, 8))

# Plot KDE for each length using updated color scheme
sns.kdeplot(df['quote_length'], label='Goodreads Sample', fill=True, 
            color=COLOR_SAMPLE_QUOTE, alpha=0.8)
sns.kdeplot(df['matched_snippet_length'], label='Google Books Length Matched', 
            fill=True, color=COLOR_MATCHED_SNIPPET, alpha=0.7)
sns.kdeplot(df['ocr_page_length'], label='Full Page', fill=True, 
            color=COLOR_PAGE_TEXT, alpha=0.5)

# Professional styling
plt.title('Distribution of Text Lengths (Word Count)', fontsize=18, fontweight='bold', 
          fontfamily='Arial', pad=20)
plt.xlabel('Number of Words', fontsize=14, fontfamily='Arial')
plt.ylabel('Density', fontsize=14, fontfamily='Arial')

# Clean legend styling
plt.legend(frameon=True, fancybox=True, shadow=False, fontsize=12, framealpha=0.9)

# Clean axes styling
ax = plt.gca()
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Clean tick styling
ax.tick_params(axis='both', which='major', labelsize=11, colors='#333333')

# Set x-limit to zoom in on the primary area of interest
plt.xlim(0, 400)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("TEXT LENGTH ANALYSIS")
print("="*60)
print(df[['quote_length', 'matched_snippet_length', 'ocr_page_length']].describe())

In [ ]:
# --- Smoothed Text Length Trends with Modern Styling ---

# Sort the dataframe by the length of the sample quote
# This creates a meaningful progression for the x-axis
df_sorted = df.sort_values(by='quote_length').reset_index(drop=True)

# Define the window size for the rolling average
# A smaller window size means less smoothing and more accurate representation
window_size = 50

# Calculate the rolling mean for each length column
df_sorted['quote_smooth'] = df_sorted['quote_length'].rolling(window=window_size).mean()
df_sorted['snippet_smooth'] = df_sorted['matched_snippet_length'].rolling(window=window_size).mean()
df_sorted['page_smooth'] = df_sorted['ocr_page_length'].rolling(window=window_size).mean()

# Modern, clean visualization with thesis color scheme
plt.style.use('default')
plt.figure(figsize=(16, 9))

# Plot the smoothed lines with updated colors and smart overlap handling
# Use different line styles and widths to distinguish overlapping lines
plt.plot(df_sorted.index, df_sorted['quote_smooth'], 
         label='Goodreads Sample', linewidth=4, 
         color=COLOR_SAMPLE_QUOTE, alpha=0.8, linestyle='-')
plt.plot(df_sorted.index, df_sorted['snippet_smooth'], 
         label='Google Books Length Matched', linewidth=2, 
         color=COLOR_MATCHED_SNIPPET, alpha=0.9, linestyle=':', 
         marker='o', markersize=1, markevery=20)
plt.plot(df_sorted.index, df_sorted['page_smooth'], 
         label='Google Books Page', linewidth=3, 
         color=COLOR_PAGE_TEXT, alpha=0.9, linestyle='--')

# Fill the area under the lines with matching colors
plt.fill_between(df_sorted.index, df_sorted['quote_smooth'], 
                alpha=0.15, color=COLOR_SAMPLE_QUOTE)
plt.fill_between(df_sorted.index, df_sorted['snippet_smooth'], 
                alpha=0.15, color=COLOR_MATCHED_SNIPPET)
plt.fill_between(df_sorted.index, df_sorted['page_smooth'], 
                alpha=0.1, color=COLOR_PAGE_TEXT)

# Professional styling
plt.title(f'Smoothed Trend of Text Lengths Across Dataset (Window Size: {window_size})', 
          fontsize=18, fontweight='bold', fontfamily='Arial', pad=20)
plt.xlabel('Quotes (Sorted by Increasing Length)', fontsize=14, fontfamily='Arial')
plt.ylabel(f'Average Number of Words (Rolling {window_size}-Quote Window)', fontsize=14, fontfamily='Arial')

# Clean legend styling
plt.legend(frameon=True, fancybox=True, shadow=False, fontsize=12, 
          loc='upper left', framealpha=0.9)

# Clean axes styling
ax = plt.gca()
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Clean tick styling
ax.tick_params(axis='both', which='major', labelsize=11, colors='#333333')

# Set y-limit to focus on the quote/snippet relationship
plt.ylim(0, 400)
plt.tight_layout()
plt.show()


In [ ]:
%pip install textstat
%pip install readability

In [ ]:
# ==============================================================================
# Cell 9: Readability Analysis (Flesch & Coleman-Liau)
# ==============================================================================
# This cell calculates, visualizes, and statistically analyzes two key
# readability metrics. It uses the global 'manager', 'palette', and
# 'sources_readability' variables defined in the Master Setup Cell.

# ───────────────────────── 0. imports ─────────────────────────
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import textstat
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ──────────────────── 1. readability metrics calculation ────────────────────
def safe_metric(func, text, min_words=10):
    if not isinstance(text, str) or len(text.split()) < min_words: return np.nan
    try: return func(text)
    except Exception: return np.nan

metrics = { "flesch": textstat.flesch_reading_ease, "cl": textstat.coleman_liau_index }

print("Calculating readability metrics...")
for src, tag in sources_readability.items():
    for m, f in metrics.items():
        if f"{tag}_{m}" not in df.columns:
            df[f"{tag}_{m}"] = df[src].progress_apply(lambda x: safe_metric(f, x))

# ────────────────── 2. long-format dataframe for plotting ──────────────────
long = pd.concat(
    df[[f"{tag}_{m}"]].rename(columns={f"{tag}_{m}": m}).assign(TextType=src)
    for src, tag in sources_readability.items()
    for m in metrics
).melt(id_vars="TextType", var_name="Metric", value_name="Score").dropna()

# ───────────────────────── 3. generate and save plots ─────────────────────────
def plot_metric(metric, title, invert=False):
    fig, ax = plt.subplots(figsize=(12, 6))
    for src in palette:
        sns.kdeplot(
            long.loc[(long.Metric == metric) & (long.TextType == src), "Score"],
            label=src, fill=True, alpha=.55, linewidth=0, color=palette[src], ax=ax
        )
    ax.set_title(title, fontsize=16, weight="bold")
    ax.set_xlabel(f"Score ({'Lower Score = Easier' if invert else 'Higher Score = Easier'})", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.legend(fontsize=9, ncol=1, loc="upper left")
    sns.despine(ax=ax); fig.tight_layout(); plt.show()
    return fig

print("\nGenerating and saving readability plots...")
fig_flesch = plot_metric("flesch", "Distribution of Flesch Reading Ease")
manager.save_plot(fig_flesch, 'readability', 'flesch_reading_ease_distribution.png')
fig_cl = plot_metric("cl", "Distribution of Coleman–Liau Index", invert=True)
manager.save_plot(fig_cl, 'readability', 'coleman_liau_index_distribution.png')

# ───────────────────────── 4. descriptive statistics ──────────────────────────
print("\n" + "="*80 + "\nDescriptive Statistics for Readability Metrics\n" + "="*80)
desc_stats = (long.groupby(["Metric", "TextType"])["Score"]
              .describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(2))
print(desc_stats)
manager.save_table(desc_stats, 'readability', 'descriptive_stats.csv')
print("="*80)

# ─────────────────── 5. inferential tests vs. baseline ───────────────────
baseline_src = "Google Books Length Matched Snippet"
quote_cols = [c for c in palette if c != baseline_src]
rows, pvals = [], []
for m in metrics:
    base_vals = long.loc[(long.Metric == m) & (long.TextType == baseline_src), "Score"].dropna()
    for q_src in quote_cols:
        q_vals = long.loc[(long.Metric == m) & (long.TextType == q_src), "Score"].dropna()
        if len(q_vals) > 0 and len(base_vals) > 0:
            alt = "greater" if m == "flesch" else "less"
            u, p = mannwhitneyu(q_vals, base_vals, alternative=alt, use_continuity=True)
            rows.append({"Metric": m, "Comparison": q_src, "U-statistic": u,
                         "p_value_raw": p, "Cliff_Delta": cliffs_delta(q_vals.values, base_vals.values)})
            pvals.append(p)

_, p_adj, _, _ = multipletests(pvals, method="fdr_bh")
inferential_results = pd.DataFrame(rows)
inferential_results["p_value_fdr_bh"] = p_adj
print("\n" + "="*80 + f"\nInferential Statistics: Mann–Whitney U vs. Baseline ('{baseline_src}')\n" + "="*80)
fmt = {"U-statistic": lambda x: f"{x:,.0f}", "p_value_raw": lambda x: f"{x:.2e}", "p_value_fdr_bh": lambda x: f"{x:.2e}" if x < 0.01 else f"{x:.3f}", "Cliff_Delta": lambda x: f"{x:+.3f}"}
print(inferential_results.to_string(index=False, formatters=fmt))
manager.save_table(inferential_results, 'readability', 'inferential_stats_vs_baseline.csv')
print("="*80)

# ────────── 6. save qualitative examples of hardest & easiest texts ──────────
print("\n" + "="*100 + "\nQUALITATIVE EXAMPLES: SAVING HARDEST AND EASIEST TEXTS\n" + "="*100)
for text_col, tag in sources_readability.items():
    flesch_col = f"{tag}_flesch"
    if text_col not in df.columns or flesch_col not in df.columns: continue
    work_df = df[[text_col, flesch_col]].copy().dropna()
    if len(work_df) < 40:
        print(f"Skipping {text_col}: Not enough data for 20/20 examples.")
        continue
    work_df_sorted = work_df.sort_values(flesch_col)
    manager.save_text_examples(
        highest_df=work_df_sorted.tail(20), lowest_df=work_df_sorted.head(20),
        metric_name='readability', filename=f"examples_flesch_{tag}.txt",
        text_col=text_col, metric_col=flesch_col, metric_unit="Flesch Score",
        low_label="Hardest to Read", high_label="Easiest to Read"
    )
print(f"\n{'=' * 100}\nReadability Analysis Complete\n{'=' * 100}")

In [ ]:
# ==============================================================================
# Cell 10: Lexical Diversity Analysis (MTLD)
# ==============================================================================
# This cell measures vocabulary richness using MTLD. It uses the global
# 'manager', 'palette', and 'sources_lex_div' variables.

# ───────────────────────── 0. imports & setup ─────────────────────────
import nltk
from lexical_diversity import lex_div as ld
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from tqdm.auto import tqdm
try: nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError: nltk.download('punkt', quiet=True)

# ────────────────── 1. lexical diversity metric calculation ─────────────────
def get_lexical_diversity(text, min_words=20):
    try:
        if pd.isna(text) or not isinstance(text, str): return np.nan
        words = [word for word in nltk.word_tokenize(str(text).lower()) if word.isalnum()]
        if len(words) < min_words: return np.nan
        return ld.mtld(words)
    except (TypeError, ValueError): return np.nan

print("Calculating lexical diversity (MTLD)...")
for src_col, metric_col in sources_lex_div.items():
    if metric_col not in df.columns:
        df[metric_col] = df[src_col].progress_apply(get_lexical_diversity)

long_lex = pd.concat([
    df[[metric_col]].rename(columns={metric_col: "Score"}).assign(TextType=src_col)
    for src_col, metric_col in sources_lex_div.items()
]).dropna()

# ───────────────────────── 2. generate and save plot ──────────────────────────
print("\nGenerating and saving lexical diversity plot...")
fig_lex, ax_lex = plt.subplots(figsize=(12, 7))
for src_name in palette.keys():
    sns.kdeplot(
        long_lex.loc[long_lex['TextType'] == src_name, "Score"],
        label=src_name, fill=True, alpha=0.55, linewidth=0,
        color=palette[src_name], ax=ax_lex
    )
ax_lex.set_title('Distribution of Lexical Diversity (MTLD)', fontsize=16, weight="bold")
ax_lex.set_xlabel('MTLD Score (Higher = More Diverse Vocabulary)', fontsize=12)
ax_lex.set_ylabel('Density', fontsize=12)
ax_lex.legend(fontsize=9, loc='upper right'); sns.despine(ax=ax_lex); fig_lex.tight_layout(); plt.show()
manager.save_plot(fig_lex, 'lexical_diversity', 'mtld_distribution.png')

# ───────────────────────── 3. descriptive statistics ──────────────────────────
print("\n" + "="*80 + "\nDescriptive Statistics for Lexical Diversity (MTLD)\n" + "="*80)
desc_stats_lex = (long_lex.groupby("TextType")["Score"]
                  .describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(2))
print(desc_stats_lex)
manager.save_table(desc_stats_lex, 'lexical_diversity', 'descriptive_stats.csv')
lex_ranking = desc_stats_lex[['mean']].sort_values(by='mean', ascending=False)
manager.save_table(lex_ranking, 'lexical_diversity', 'mean_mtld_ranking.csv')
print("="*80)

# ───────── 4. save qualitative examples of highest & lowest diversity ─────────
print("\n" + "="*100 + "\nQUALITATIVE EXAMPLES: SAVING HIGHEST AND LOWEST DIVERSITY TEXTS\n" + "="*100)
for text_col, metric_col in sources_lex_div.items():
    if text_col not in df.columns or metric_col not in df.columns: continue
    work_df = df[[text_col, metric_col]].copy().dropna()
    if len(work_df) < 40:
        print(f"Skipping {text_col}: Not enough data for 20/20 examples.")
        continue
    work_df_sorted = work_df.sort_values(metric_col)
    manager.save_text_examples(
        highest_df=work_df_sorted.tail(20), lowest_df=work_df_sorted.head(20),
        metric_name='lexical_diversity',
        filename=f"examples_mtld_{metric_col.replace('_lex_div', '')}.txt",
        text_col=text_col, metric_col=metric_col, metric_unit="MTLD Score",
        low_label="Least Diverse Vocabulary", high_label="Most Diverse Vocabulary"
    )
print(f"\n{'=' * 100}\nLexical Diversity Analysis Complete\n{'=' * 100}")

In [ ]:
# ==============================================================================
# Cell 13: Linguistic Predictability (GPT-2 Surprisal) - PUBLICATION-READY PLOT
# ==============================================================================
# This version creates a more scientifically robust plot by ordering categories
# by median, showing the median/IQR with an embedded boxplot, and improving
# the clarity of the raw data points.

# ───────────────────────── 0. imports & setup ─────────────────────────
import torch
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings

# Define the source text columns and their corresponding metric column names
sources_surprisal = {
    "Goodreads Sample Quote": "sample_surprisal",
    "Goodreads Popular Quote": "popular_surprisal",
    "Google Books Length Matched Snippet": "snippet_surprisal",
    "T50 Quote": "t50_surprisal",
    "T50 Quote-Free Context Length Matched": "t50freelength_surprisal",
    "Non-Literary Baseline": "nonlit_surprisal"
}

# Check if all surprisal columns have been calculated. If so, skip the cell.
# This assumes 'df' is loaded and available from a previous cell.
if all(col in df.columns for col in sources_surprisal.values()):
    print("✓ All surprisal scores already calculated. Skipping GPT-2 analysis cell.")
else:
    # ─────────────────── 1. Load Model and Set Device ───────────────────
    print("Initializing Surprisal analysis...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_name = 'gpt2'
    print(f"Using device: {device}")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
        tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
    model.eval()
    print(f"✓ GPT-2 model and tokenizer loaded successfully.")

    # ───────────────────── 2. Surprisal Helper Function ─────────────────────
    def get_surprisal(text):
        if pd.isna(text) or not isinstance(text, str) or len(text.strip()) == 0: return np.nan
        try:
            max_length, stride, lls = tokenizer.model_max_length, 256, []
            with torch.no_grad():
                for i in range(0, len(text), stride):
                    begin_loc, end_loc = max(i + stride - max_length, 0), min(i + stride, len(text))
                    trg_len = end_loc - i
                    input_ids = tokenizer.encode(text[begin_loc:end_loc], return_tensors="pt").to(device)
                    target_ids = input_ids.clone(); target_ids[:, :-trg_len] = -100
                    outputs = model(input_ids, labels=target_ids)
                    lls.append(outputs.loss * trg_len)
            return torch.stack(lls).sum().item() / end_loc
        except Exception: return np.nan

    # ─────────────────────── 3. Calculate Surprisal Scores ────────────────────────
    print("\nCleaning up existing surprisal columns to ensure they are numeric...")
    for metric_col in sources_surprisal.values():
        if metric_col in df.columns and df[metric_col].dtype == 'object':
            df[metric_col] = df[metric_col].apply(lambda x: x.item() if hasattr(x, 'item') else x)
            df[metric_col] = pd.to_numeric(df[metric_col], errors='coerce')

    print("\nCalculating GPT-2 surprisal scores (if needed)...")
    for src_col, metric_col in sources_surprisal.items():
        if metric_col not in df.columns:
            print(f"Processing column: {src_col}")
            tqdm.pandas(desc=f"Surprisal for {src_col}")
            df[metric_col] = df[src_col].progress_apply(get_surprisal)
        else:
            print(f"Skipping {src_col}, results already exist.")

    # ─────────────────── 4. Visualization ───────────────────
    print("\nGenerating and saving enhanced surprisal plot…")
    long_surp = pd.concat([
        df[[metric_col]].rename(columns={metric_col: "Surprisal"}).assign(TextType=src_col)
        for src_col, metric_col in sources_surprisal.items()
    ], ignore_index=True)
    long_surp["Surprisal"] = pd.to_numeric(long_surp["Surprisal"], errors='coerce')
    long_surp.dropna(subset=["Surprisal"], inplace=True)

    # --- IMPROVEMENT 1: Order categories by median surprisal ---
    median_order = long_surp.groupby('TextType')['Surprisal'].median().sort_values().index
    order = median_order.tolist()

    # --- IMPROVEMENT 2: Finer control over plot aesthetics ---
    dot_kws = dict(color="black", alpha=0.25, jitter=0.3, size=1.5, zorder=3)
    vio_kws = dict(alpha=1, cut=0, bw_adjust=.8, inner=None, linewidth=0)

    fig, ax = plt.subplots(figsize=(12, 7))

    # Layer 1: Draw translucent violins
    sns.violinplot(data=long_surp, y="TextType", x="Surprisal", order=order, palette=palette, zorder=1, **vio_kws, ax=ax)
    # Layer 2: Overlay semi-transparent data points
    sns.stripplot(data=long_surp, y="TextType", x="Surprisal", order=order, **dot_kws, ax=ax)

    # --- IMPROVEMENT 3: Add a narrow boxplot for Median/IQR instead of just a mean line ---
    sns.boxplot(
        data=long_surp, y="TextType", x="Surprisal", order=order, ax=ax,
        showfliers=False,         # The stripplot shows all points
        width=0.15,               # Make the box narrow
        boxprops={'facecolor':'white', 'edgecolor':'black', 'zorder': 5},
        medianprops={'color':'black', 'linewidth': 2, 'zorder': 6},
        whiskerprops={'color':'black', 'zorder': 6, 'linewidth': 1.5},
        capprops={'color':'black', 'zorder': 6, 'linewidth': 1.5}
    )

    # Final Aesthetics
    ax.set_title("Distribution of Linguistic Predictability (GPT-2 Surprisal)", fontsize=16, weight="bold")
    ax.set_xlabel("GPT-2 Surprisal (Higher = Less Predictable)")
    ax.set_ylabel("")
    ax.set_yticklabels(order, fontsize=10) # Use the new sorted order
    sns.despine(ax=ax, left=True)
    if ax.get_legend() is not None:
        ax.legend_.remove()
    fig.tight_layout()

    plt.show()
    # Note: Assumes 'manager' object is defined in a previous cell
    manager.save_plot(fig, "surprisal", "gpt2_surprisal_final.png")
    print("✓ Final plot saved.")

    # ───────────────────────── 5. Statistics & Saving ───────────────────────────
    print("\n" + "="*80 + "\nDescriptive Statistics for GPT-2 Surprisal\n" + "="*80)
    desc_stats_surp = (long_surp.groupby("TextType")["Surprisal"].describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(3))
    print(desc_stats_surp)
    manager.save_table(desc_stats_surp, 'surprisal', 'descriptive_stats.csv')
    baseline_src = "Google Books Length Matched Snippet"
    quote_cols = [c for c in sources_surprisal.keys() if c != baseline_src]
    rows_surp, pvals_surp = [], []
    base_vals_surp = df[sources_surprisal[baseline_src]].dropna().astype(float)
    for q_src in quote_cols:
        q_vals = df[sources_surprisal[q_src]].dropna().astype(float)
        if len(q_vals) > 0 and len(base_vals_surp) > 0:
            u, p = mannwhitneyu(q_vals, base_vals_surp, alternative="greater")
            # Note: Assumes 'cliffs_delta' is defined in a previous cell
            rows_surp.append({"Comparison": q_src, "U-statistic": u, "p_value_raw": p, "Cliff_Delta": cliffs_delta(q_vals.values, base_vals_surp.values)})
            pvals_surp.append(p)
    if pvals_surp:
        _, p_adj, _, _ = multipletests(pvals_surp, method="fdr_bh")
        inferential_results_surp = pd.DataFrame(rows_surp)
        inferential_results_surp["p_value_fdr_bh"] = p_adj
        print("\n" + "="*80 + f"\nInferential Statistics: Mann–Whitney U vs. Baseline ('{baseline_src}')\n" + "="*80)
        fmt = {"U-statistic": lambda x: f"{x:,.0f}", "p_value_raw": lambda x: f"{x:.2e}", "p_value_fdr_bh": lambda x: f"{x:.2e}" if x < 0.01 else f"{x:.3f}", "Cliff_Delta": lambda x: f"{x:+.3f}"}
        print(inferential_results_surp.to_string(index=False, formatters=fmt))
        manager.save_table(inferential_results_surp, 'surprisal', 'inferential_stats_vs_baseline.csv')

    # ───────── 6. Save Qualitative Examples ─────────
    print("\n" + "="*100 + "\nQUALITATIVE EXAMPLES: SAVING HIGHEST AND LOWEST SURPRISAL TEXTS\n" + "="*100)
    for text_col, metric_col in sources_surprisal.items():
        work_df = df[[text_col, metric_col]].copy().dropna()
        if len(work_df) < 40: continue
        work_df_sorted = work_df.sort_values(metric_col)
        manager.save_text_examples(
            highest_df=work_df_sorted.tail(20), lowest_df=work_df_sorted.head(20),
            metric_name='surprisal', filename=f"examples_surprisal_{metric_col.replace('_surprisal', '')}.txt",
            text_col=text_col, metric_col=metric_col, metric_unit="GPT-2 Surprisal",
            low_label="Most Predictable", high_label="Most Surprising"
        )
    print(f"\n{'=' * 100}\nGPT-2 Surprisal Analysis Complete\n{'=' * 100}")


In [ ]:
# ==============================================================================
# Cell 11: Information Content Analysis (Shannon Entropy)
# ==============================================================================
# This cell measures word-level Shannon Entropy. It uses the global 'manager',
# 'palette', 'sources_entropy', and 'cliffs_delta' variables.

# ───────────────────────── 0. imports & setup ─────────────────────────
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import nltk, collections
from tqdm.auto import tqdm
from scipy.stats import entropy, mannwhitneyu
from statsmodels.stats.multitest import multipletests
try: nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError: nltk.download('punkt', quiet=True)

# ───────────────────── 1. entropy metric calculation ────────────────────
def shannon_entropy(text, min_words=10):
    if not isinstance(text, str) or pd.isna(text): return np.nan
    tokens = [w.lower() for w in nltk.word_tokenize(text) if w.isalnum()]
    if len(tokens) < min_words: return np.nan
    counts = collections.Counter(tokens)
    probabilities = np.array(list(counts.values()), dtype=float) / len(tokens)
    return entropy(probabilities, base=2)

print("Calculating Shannon entropy...")
for src_col, metric_col in sources_entropy.items():
    if metric_col not in df.columns:
        df[metric_col] = df[src_col].progress_apply(shannon_entropy)

long_ent = pd.concat([
    df[[metric_col]].rename(columns={metric_col: "Entropy"}).assign(TextType=src_col)
    for src_col, metric_col in sources_entropy.items()
]).dropna()

# ───────────────────────── 2. generate and save plot ──────────────────────────
print("\nGenerating and saving entropy plot...")
fig_ent, ax_ent = plt.subplots(figsize=(12, 6))
for src in palette:
    sns.kdeplot(
        long_ent.loc[long_ent.TextType == src, "Entropy"],
        label=src, fill=True, alpha=.55, linewidth=0, color=palette[src], ax=ax_ent
    )
ax_ent.set_title("Distribution of Word-Level Shannon Entropy", fontsize=16, weight="bold")
ax_ent.set_xlabel("Entropy (bits) – Higher = More Unpredictable", fontsize=12)
ax_ent.set_ylabel("Density", fontsize=12)
ax_ent.legend(fontsize=9, loc="upper left"); sns.despine(ax=ax_ent); fig_ent.tight_layout(); plt.show()
manager.save_plot(fig_ent, 'entropy', 'shannon_entropy_distribution.png')

# ───────────────────────── 3. descriptive statistics ────────────────────────
print("\n" + "="*80 + "\nDescriptive Statistics for Shannon Entropy\n" + "="*80)
desc_stats_ent = (long_ent.groupby("TextType")["Entropy"]
                  .describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(2))
print(desc_stats_ent)
manager.save_table(desc_stats_ent, 'entropy', 'descriptive_stats.csv')
print("="*80)

# ─────────────────── 4. inferential tests vs. baseline ────────────────────
baseline_src = "Google Books Length Matched Snippet"
quote_cols = [c for c in palette if c != baseline_src]
rows_ent, pvals_ent = [], []
base_vals_ent = long_ent.loc[long_ent.TextType == baseline_src, "Entropy"].dropna()
for q_src in quote_cols:
    q_vals = long_ent.loc[long_ent.TextType == q_src, "Entropy"].dropna()
    if len(q_vals) > 0 and len(base_vals_ent) > 0:
        u, p = mannwhitneyu(q_vals, base_vals_ent, alternative="greater")
        rows_ent.append({"Comparison": q_src, "U-statistic": u,
                         "p_value_raw": p, "Cliff_Delta": cliffs_delta(q_vals.values, base_vals_ent.values)})
        pvals_ent.append(p)

_, p_adj, _, _ = multipletests(pvals_ent, method="fdr_bh")
inferential_results_ent = pd.DataFrame(rows_ent)
inferential_results_ent["p_value_fdr_bh"] = p_adj
print("\n" + "="*80 + f"\nInferential Statistics: Mann–Whitney U vs. Baseline ('{baseline_src}')\n" + "="*80)
fmt = {"U-statistic": lambda x: f"{x:,.0f}", "p_value_raw": lambda x: f"{x:.2e}", "p_value_fdr_bh": lambda x: f"{x:.2e}" if x < 0.01 else f"{x:.3f}", "Cliff_Delta": lambda x: f"{x:+.3f}"}
print(inferential_results_ent.to_string(index=False, formatters=fmt))
manager.save_table(inferential_results_ent, 'entropy', 'inferential_stats_vs_baseline.csv')
print("="*80)

# ────────── 5. save qualitative examples of highest & lowest entropy ──────────
print("\n" + "="*100 + "\nQUALITATIVE EXAMPLES: SAVING HIGHEST AND LOWEST ENTROPY TEXTS\n" + "="*100)
for text_col, metric_col in sources_entropy.items():
    if text_col not in df.columns or metric_col not in df.columns: continue
    work_df = df[[text_col, metric_col]].copy().dropna()
    if len(work_df) < 40:
        print(f"Skipping {text_col}: Not enough data for 20/20 examples.")
        continue
    work_df_sorted = work_df.sort_values(metric_col)
    manager.save_text_examples(
        highest_df=work_df_sorted.tail(20), lowest_df=work_df_sorted.head(20),
        metric_name='entropy',
        filename=f"examples_entropy_{metric_col.replace('_entropy', '')}.txt",
        text_col=text_col, metric_col=metric_col, metric_unit="Shannon Entropy (bits)",
        low_label="Most Predictable Word Usage", high_label="Most Unpredictable Word Usage"
    )
print(f"\n{'=' * 100}\nShannon Entropy Analysis Complete\n{'=' * 100}")

In [ ]:
# ==============================================================================
# Cell 12: Lexical Novelty (Δ-PMI) - Final Plot with Dynamic Title
# ==============================================================================
# This cell integrates the delta-PMI pipeline. The visualization is a vertical
# distribution plot with outliers removed and a dynamic title stating the
# exact percentage of data trimmed for clarity.

# ───────────────────────── 0. imports & setup ─────────────────────────
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import spacy, collections, pickle, re, os
from itertools import chain
from tqdm.auto import tqdm
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ──────────────────── 1. Δ-PMI helper functions ────────────────────
TOKEN_RE = re.compile(r"^[a-z']+$", re.I)
def iter_lemmas(nlp, texts):
    for doc in nlp.pipe(texts, batch_size=250): yield [t.lemma_.lower() for t in doc if t.is_alpha and TOKEN_RE.match(t.lemma_)]
def sliding_pairs(tokens, width=5):
    for i, w in enumerate(tokens):
        for j in range(max(0, i - width), min(len(tokens), i + width + 1)):
            if i != j: yield w, tokens[j]
def collect_counts(token_stream, desc):
    uni, bi = collections.Counter(), collections.Counter()
    for toks in tqdm(token_stream, desc=desc): uni.update(toks); bi.update(sliding_pairs(toks))
    return uni, bi
def delta_pmi(tokens, bg_uni, bg_bi, N_bg, alpha=0.5):
    if not isinstance(tokens, list) or len(tokens) < 2: return np.nan
    doc_uni, doc_bi, N_doc = collections.Counter(tokens), collections.Counter(sliding_pairs(tokens)), len(tokens)
    vals = []
    for (w1, w2), c12 in doc_bi.items():
        if w1 not in bg_uni or w2 not in bg_uni: continue
        p_doc_12, p_doc_1, p_doc_2 = c12 / N_doc, doc_uni[w1] / N_doc, doc_uni[w2] / N_doc
        pmi_doc = np.log2(p_doc_12 / (p_doc_1 * p_doc_2))
        p_bg_12 = (bg_bi.get((w1, w2), 0) + alpha) / (N_bg + alpha)
        p_bg_1, p_bg_2 = bg_uni[w1] / N_bg, bg_uni[w2] / N_bg
        pmi_bg = np.log2(p_bg_12 / (p_bg_1 * p_bg_2))
        vals.append(pmi_doc - pmi_bg)
    return np.mean(vals) if vals else np.nan

# ────────────────── 2. Build or Load Background Model ──────────────────
print("Initializing Δ-PMI analysis...")
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
pmi_path = manager._get_metric_path('delta_pmi')
bg_model_path = os.path.join(pmi_path, 'bg_counters.pkl')
if os.path.exists(bg_model_path):
    print(f"Loading cached background PMI model from: {bg_model_path}")
    with open(bg_model_path, 'rb') as f: bg_model = pickle.load(f)
    bg_uni, bg_bi, N_bg = bg_model['uni'], bg_model['bi'], bg_model['N']
else:
    print("Building background PMI model (this may take a while)...")
    bg_cols = ["Google Books Page Text", "T50 Quote-Free Context", "Non-Literary Baseline"]
    bg_texts = chain.from_iterable(df[col].fillna("").tolist() for col in bg_cols)
    bg_uni, bg_bi = collect_counts(iter_lemmas(nlp, bg_texts), desc="Counting BG Frequencies")
    N_bg = sum(bg_uni.values())
    with open(bg_model_path, 'wb') as f: pickle.dump({"uni": bg_uni, "bi": bg_bi, "N": N_bg}, f)
    print(f"✓ Background PMI model built and saved to {bg_model_path}")
print(f"✓ Background model ready: {N_bg:,} tokens, {len(bg_uni):,} types.")

# ────────────────────── 3. Calculate Δ-PMI Scores ───────────────────────
sources_pmi = { "Goodreads Sample Quote": "sample_pmi", "Goodreads Popular Quote": "popular_pmi", "Google Books Length Matched Snippet": "snippet_pmi", "T50 Quote": "t50_pmi", "T50 Quote-Free Context Length Matched": "t50freelength_pmi" }
print("\nCalculating Δ-PMI scores for target columns...")
for src_col, metric_col in sources_pmi.items():
    if metric_col not in df.columns:
        lemmatized_texts = list(iter_lemmas(nlp, df[src_col].fillna("")))
        df[metric_col] = pd.Series(lemmatized_texts).progress_apply(lambda toks: delta_pmi(toks, bg_uni, bg_bi, N_bg))

# ──── 4. Visualization (Vertical Distribution Plot with Dynamic Title) ────
print("\nGenerating and saving Δ-PMI Vertical Distribution Plot...")
long_pmi = pd.concat([
    df[[metric_col]].rename(columns={metric_col: "Delta_PMI"}).assign(TextType=src_col)
    for src_col, metric_col in sources_pmi.items()
]).dropna()

# --- Outlier removal for visualization only ---
Q1 = long_pmi['Delta_PMI'].quantile(0.25)
Q3 = long_pmi['Delta_PMI'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
original_count = len(long_pmi)
long_pmi_filtered = long_pmi[(long_pmi['Delta_PMI'] >= lower_bound) & (long_pmi['Delta_PMI'] <= upper_bound)]
removed_count = original_count - len(long_pmi_filtered)
removed_percent = (removed_count / original_count) * 100
print(f"Visual Clarity Report: Removed {removed_count} outliers ({removed_percent:.1f}%) for plotting.")

fig_pmi, ax_pmi = plt.subplots(figsize=(12, 8))
ax_pmi.axhline(0, color='black', linestyle='--', linewidth=1.2, zorder=0)

sns.violinplot(
    x="TextType", y="Delta_PMI", data=long_pmi_filtered,
    order=sources_pmi.keys(), palette=palette, ax=ax_pmi,
    inner=None, cut=0, linewidth=1.5
)
sns.pointplot(
    x="TextType", y="Delta_PMI", data=long_pmi_filtered,
    order=sources_pmi.keys(), estimator=np.mean, join=False,
    color='white', markers='o', scale=0.9, errorbar=None, ax=ax_pmi
)

# A clean, concise title is best.
ax_pmi.set_title("Lexical Novelty (Δ-PMI) Across Text Types", fontsize=16, weight="bold")
ax_pmi.set_xlabel("")
ax_pmi.set_ylabel("Δ-PMI Score (Higher = More Novel)")
ax_pmi.set_xticklabels(ax_pmi.get_xticklabels(), rotation=15, ha="right")
ax_pmi.grid(axis='y', linestyle=':', alpha=0.6)
sns.despine(ax=ax_pmi)
fig_pmi.tight_layout()
plt.show()
manager.save_plot(fig_pmi, 'delta_pmi', 'delta_pmi_distribution_vertical_no_outliers.png')

# ────────── 5. Statistics & Saving (on ORIGINAL data) ──────────
print("\n" + "="*80 + "\nDescriptive Statistics for Δ-PMI (Full Dataset)\n" + "="*80)
desc_stats_pmi = (long_pmi.groupby("TextType")["Delta_PMI"].describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(3))
print(desc_stats_pmi)
manager.save_table(desc_stats_pmi, 'delta_pmi', 'descriptive_stats_full.csv')
baseline_src = "Google Books Length Matched Snippet"
quote_cols = [c for c in sources_pmi.keys() if c != baseline_src]
rows_pmi, pvals_pmi = [], []
base_vals_pmi = df[sources_pmi[baseline_src]].dropna()
for q_src in quote_cols:
    q_vals = df[sources_pmi[q_src]].dropna()
    if len(q_vals) > 0 and len(base_vals_pmi) > 0:
        u, p = mannwhitneyu(q_vals, base_vals_pmi, alternative="greater")
        rows_pmi.append({"Comparison": q_src, "U-statistic": u, "p_value_raw": p, "Cliff_Delta": cliffs_delta(q_vals.values, base_vals_pmi.values)})
        pvals_pmi.append(p)
_, p_adj, _, _ = multipletests(pvals_pmi, method="fdr_bh")
inferential_results_pmi = pd.DataFrame(rows_pmi); inferential_results_pmi["p_value_fdr_bh"] = p_adj
print("\n" + "="*80 + f"\nInferential Statistics (Full Dataset): Mann–Whitney U vs. Baseline ('{baseline_src}')\n" + "="*80)
fmt = {"U-statistic": lambda x: f"{x:,.0f}", "p_value_raw": lambda x: f"{x:.2e}", "p_value_fdr_bh": lambda x: f"{x:.2e}" if x < 0.01 else f"{x:.3f}", "Cliff_Delta": lambda x: f"{x:+.3f}"}
print(inferential_results_pmi.to_string(index=False, formatters=fmt))
manager.save_table(inferential_results_pmi, 'delta_pmi', 'inferential_stats_vs_baseline_full.csv')

# ───────── 6. Save Qualitative Examples (from ORIGINAL data) ─────────
print("\n" + "="*100 + "\nQUALITATIVE EXAMPLES: SAVING HIGHEST AND LOWEST Δ-PMI TEXTS\n" + "="*100)
for text_col, metric_col in sources_pmi.items():
    work_df = df[[text_col, metric_col]].copy().dropna()
    if len(work_df) < 40: continue
    work_df_sorted = work_df.sort_values(metric_col)
    manager.save_text_examples(
        highest_df=work_df_sorted.tail(20), lowest_df=work_df_sorted.head(20),
        metric_name='delta_pmi', filename=f"examples_pmi_{metric_col.replace('_pmi', '')}.txt",
        text_col=text_col, metric_col=metric_col, metric_unit="Δ-PMI Score",
        low_label="Most Conventional Word Usage", high_label="Most Novel Word Usage"
    )
print(f"\n{'=' * 100}\nΔ-PMI Analysis Complete\n{'=' * 100}")

In [ ]:
# ==============================================================================
# Cell 15: Affective Analysis (Corrected Version)
# ==============================================================================
# This version uses a more robust method to skip previously computed results
# by checking for the existence of columns directly in the DataFrame.

# ─────────────────────── 0. Imports & setup ───────────────────────
import torch, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import time
import pathlib

# ───────────────── 1. Model / tokenizer (loaded once) ─────────────────
print("Loading sentiment-analysis model …")
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
sentiment_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if device.type == "cuda" else -1, top_k=None)
print(f"✓  Model ready  (device = {device})")

# ──────────────── 2. Simplified & Robust Helper Function ────────────────
def process_sentiment_batch(text_series: pd.Series, batch_size: int = 32) -> pd.DataFrame:
    """
    Run sentiment analysis and return a DataFrame with scores.
    Relies on the pipeline's robust, built-in truncation.
    """
    text_list = text_series.fillna("").astype(str).tolist()
    results = sentiment_pipe(text_list, batch_size=batch_size, truncation=True, padding=True, max_length=512)
    rows = []
    for res in results:
        scores = {d["label"].lower(): d["score"] for d in res}
        rows.append({
            "label": max(scores, key=scores.get), "pos": scores.get("positive", 0.0),
            "neu": scores.get("neutral", 0.0), "neg": scores.get("negative", 0.0)
        })
    return pd.DataFrame(rows)

# ─────────── 3. Compute sentiment scores & polarity (with incremental saving) ───────────
sources_sent = {
    "Goodreads Sample Quote": "sample", "Goodreads Popular Quote": "popular", "Google Books Length Matched Snippet": "snippet", "T50 Quote": "t50", "T50 Quote-Free Context Length Matched": "t50freelength", "Non-Literary Baseline": "nonlit"
}
print("\nCalculating sentiment for each text type …")
total_steps = len(sources_sent)
for i, (src_col, prefix) in enumerate(tqdm(sources_sent.items(), desc="Overall Progress")):
    step_num = i + 1
    polarity_col = f"{prefix}_sentiment_polarity"
    
    # NEW ROBUST CHECK: Skip only if the final polarity column already exists in the DataFrame.
    if polarity_col in df.columns:
        print(f"\n[Step {step_num}/{total_steps}] ✓ Skipping cached: '{src_col}'")
        continue

    print(f"\n[Step {step_num}/{total_steps}] Processing '{src_col}'...")
    start_time = time.time()
    
    sent_df = process_sentiment_batch(df[src_col])
    
    end_time = time.time()
    duration = end_time - start_time
    print(f"[Step {step_num}/{total_steps}] ✓ Completed '{src_col}' in {duration:.1f} seconds.")
    
    sent_df.columns = [f"{prefix}_sentiment_label", f"{prefix}_sentiment_pos", f"{prefix}_sentiment_neu", f"{prefix}_sentiment_neg"]
    sent_df[polarity_col] = sent_df[f"{prefix}_sentiment_pos"] - sent_df[f"{prefix}_sentiment_neg"]
    df = pd.concat([df, sent_df.set_index(df.index)], axis=1)
    
    # --- INCREMENTAL SAVE ---
    # Save the master dataframe after each successful step.
    manager.save_master_df(df)
    print(f"[Step {step_num}/{total_steps}] Progress saved to disk.")

print("\n✓  Sentiment and polarity analysis complete.")


# ──────────── 4. Visualisation 1: Label Proportions (Sorted & Improved Legend) ────────────
label_summary = pd.DataFrame({
    src: df[f"{pre}_sentiment_label"].value_counts(normalize=True)
    for src, pre in sources_sent.items()
}).T.fillna(0)

label_summary_sorted = label_summary.sort_values(by='negative', ascending=False)
sent_colors = {"negative": "#D62728", "neutral": "#808080", "positive": "#2CA02C"}
fig_bar, ax_bar = plt.subplots(figsize=(12, 7))
label_summary_sorted[["negative", "neutral", "positive"]].plot(
    kind="bar", stacked=True, color=sent_colors, ax=ax_bar, width=0.8,
    edgecolor='white', linewidth=0.7
)
ax_bar.set_title("Sentiment-Label Proportions Across Text Types", fontsize=16, weight="bold")
ax_bar.set_ylabel("Proportion")
ax_bar.set_xlabel("")
ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=15, ha="right", fontsize=11)
ax_bar.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax_bar.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., frameon=False, fontsize=11)
sns.despine(ax=ax_bar)
fig_bar.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()
manager.save_plot(fig_bar, "sentiment", "sentiment_label_proportions_sorted.png")
manager.save_table(label_summary_sorted, "sentiment", "sentiment_label_proportions_sorted.csv")

# ────────── 5. Visualisation 2: Polarity Raincloud Plot ───────────
print("\nGenerating raincloud plot for sentiment polarity...")
long_polarity = pd.concat([
    df[[f"{pre}_sentiment_polarity"]].rename(columns={f"{pre}_sentiment_polarity": "Polarity"}).assign(TextType=src)
    for src, pre in sources_sent.items()
]).dropna()
median_order = long_polarity.groupby('TextType')['Polarity'].median().sort_values().index
fig_rain, ax_rain = plt.subplots(figsize=(12, 7))
sns.violinplot(data=long_polarity, y="TextType", x="Polarity", order=median_order, palette=palette, ax=ax_rain, inner=None, cut=0, linewidth=0, alpha=0.45)
sns.stripplot(data=long_polarity, y="TextType", x="Polarity", order=median_order, ax=ax_rain, color="black", alpha=0.25, jitter=0.3, size=1.5, zorder=3)
sns.boxplot(data=long_polarity, y="TextType", x="Polarity", order=median_order, ax=ax_rain, showfliers=False, width=0.15,
            boxprops={'facecolor':'white', 'edgecolor':'black', 'zorder': 5}, medianprops={'color':'black', 'linewidth': 2, 'zorder': 6},
            whiskerprops={'color':'black', 'zorder': 6, 'linewidth': 1.5}, capprops={'color':'black', 'zorder': 6, 'linewidth': 1.5})
ax_rain.set_title("Distribution of Sentiment Polarity", fontsize=16, weight="bold")
ax_rain.set_xlabel("Sentiment Polarity Score (Positive − Negative)")
ax_rain.set_ylabel(""); sns.despine(ax=ax_rain, left=True)
ax_rain.axvline(0, color='grey', linestyle='--', linewidth=1, zorder=0)
if ax_rain.get_legend() is not None: ax_rain.get_legend().remove()
fig_rain.tight_layout(); plt.show()
manager.save_plot(fig_rain, "sentiment", "sentiment_polarity_raincloud.png")

# ──────────── 6. Descriptive & Inferential Statistics ────────────
print("\n" + "="*80 + "\nDescriptive Statistics for Sentiment Polarity\n" + "="*80)
desc_stats_pol = (long_polarity.groupby("TextType")["Polarity"].describe()[["count", "mean", "std", "25%", "50%", "75%"]].round(3))
print(desc_stats_pol)
manager.save_table(desc_stats_pol, 'sentiment', 'descriptive_stats_polarity.csv')

baseline_src = "Google Books Length Matched Snippet"
quote_cols = [c for c in sources_sent.keys() if c != baseline_src]
rows_sent, pvals_sent = [], []
base_vals_sent = long_polarity.loc[long_polarity.TextType == baseline_src, "Polarity"].dropna()
for q_src in quote_cols:
    q_vals = long_polarity.loc[long_polarity.TextType == q_src, "Polarity"].dropna()
    if len(q_vals) > 0 and len(base_vals_sent) > 0:
        u, p = mannwhitneyu(q_vals, base_vals_sent, alternative="two-sided")
        rows_sent.append({"Comparison": q_src, "U-statistic": u, "p_value_raw": p, "Cliff_Delta": cliffs_delta(q_vals.values, base_vals_sent.values)})
        pvals_sent.append(p)
if pvals_sent:
    _, p_adj, _, _ = multipletests(pvals_sent, method="fdr_bh")
    inferential_results_sent = pd.DataFrame(rows_sent); inferential_results_sent["p_value_fdr_bh"] = p_adj
    print("\n" + "="*80 + f"\nInferential Statistics: Mann–Whitney U vs. Baseline ('{baseline_src}')\n" + "="*80)
    fmt = {"U-statistic": lambda x: f"{x:,.0f}", "p_value_raw": lambda x: f"{x:.2e}", "p_value_fdr_bh": lambda x: f"{x:.2e}" if x < 0.01 else f"{x:.3f}", "Cliff_Delta": lambda x: f"{x:+.3f}"}
    print(inferential_results_sent.to_string(index=False, formatters=fmt))
    manager.save_table(inferential_results_sent, 'sentiment', 'inferential_stats_vs_baseline.csv')

# ────────────── 7. Save Qualitative Examples ───────────────
print("\n" + "="*100 + "\nQUALITATIVE EXAMPLES: SAVING HIGHEST AND LOWEST POLARITY TEXTS\n" + "="*100)
for text_col, prefix in sources_sent.items():
    metric_col = f"{prefix}_sentiment_polarity"
    work_df = df[[text_col, metric_col]].copy().dropna()
    if len(work_df) < 40: continue
    work_df_sorted = work_df.sort_values(metric_col)
    manager.save_text_examples(
        highest_df=work_df_sorted.tail(20), lowest_df=work_df_sorted.head(20),
        metric_name='sentiment', filename=f"examples_polarity_{prefix}.txt",
        text_col=text_col, metric_col=metric_col, metric_unit="Sentiment Polarity",
        low_label="Most Negative", high_label="Most Positive"
    )
print(f"\n{'=' * 100}\nSentiment Analysis Complete\n{'=' * 100}")


In [ ]:
# ==============================================================================
# Cell 16: Forward Flow Integration
# ==============================================================================
# This cell loads the pre-calculated Forward Flow (FF) metrics, transforms
# the data, and merges it into the main 'df' DataFrame.

import pathlib
import os
print("Integrating Forward Flow metrics...")

# --- 1. Define Paths and Mappings ---
# ROBUST PATHING FIX: Instead of a relative path that depends on the
# notebook's working directory, we'll find the project root and build an
# absolute path from there. This avoids pathing errors.

def find_project_root(anchor_file='requirements.txt'):
    """Finds the project root by searching upwards for an anchor file."""
    path = pathlib.Path.cwd()
    while path != path.parent:
        if (path / anchor_file).is_file():
            return path
        path = path.parent
    # Fallback if anchor is not found
    raise FileNotFoundError(f"Could not find project root containing '{anchor_file}'.")

try:
    PROJECT_ROOT = find_project_root()
    ff_metrics_path = PROJECT_ROOT / 'Analysis' / 'Outputs' / 'forward_flow' / 'forward_flow_metrics.csv'
    print(f"Project root found: {PROJECT_ROOT}")
    print(f"Constructed absolute path to metrics: {ff_metrics_path}")

    # Reuse the mapping from the readability cell (Cell 10)
    ff_column_map = {
        source_name: f"{prefix}_ff"
        for source_name, prefix in sources_readability.items()
    }

    # --- 2. Load and Reshape Forward Flow Data ---
    # Load the long-format data
    ff_df_long = pd.read_csv(ff_metrics_path)
    
    # Pivot the data from long to wide format
    ff_df_wide = ff_df_long.pivot(
        index='original_index',
        columns='source',
        values='score'
    )
    
    # --- 3. Rename Columns and Merge ---
    ff_df_wide.rename(columns=ff_column_map, inplace=True)
    
    # Merge the FF scores into the main DataFrame
    df = df.merge(ff_df_wide, left_index=True, right_index=True, how='left')
    
    print("✓ Forward Flow metrics successfully merged into the main DataFrame.")
    
    # --- 4. Display Head of New Columns ---
    ff_cols = [col for col in df.columns if col.endswith('_ff')]
    print("\nHead of the newly added Forward Flow columns:")
    print(df[ff_cols].head())

except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Please ensure 'requirements.txt' is in your project root and 'analyze_forward_flow.py' has been run.")
except Exception as e:
    print(f"An error occurred during Forward Flow integration: {e}")

In [ ]:
# ==============================================================================
# Cell 17: Stepwise Distance (SWD) Integration (Pre-Calculated)
# ==============================================================================
# This cell loads pre-calculated Stepwise Distance (SWD) metrics from the
# cache file generated by 'generate_swd_stats.py' and merges them into the
# main 'df' DataFrame. This follows the same pattern as the Forward Flow cell.

import pathlib
import pandas as pd

print("Integrating pre-calculated Stepwise Distance (SWD) metrics...")

# --- 1. Define Paths and Mappings ---
# Reusing the robust pathing function from the FF cell.
def find_project_root(anchor_file='requirements.txt'):
    """Finds the project root by searching upwards for an anchor file."""
    path = pathlib.Path.cwd()
    while path != path.parent:
        if (path / anchor_file).is_file():
            return path
        path = path.parent
    raise FileNotFoundError(f"Could not find project root containing '{anchor_file}'.")

# This map defines the columns we expect to find in the cache file.
# It's the same map used in the 'generate_swd_stats.py' script.
swd_column_map = {
    'Goodreads Sample Quote': 'sample_swd',
    'Goodreads Popular Quote': 'popular_swd',
    'Google Books Length Matched Snippet': 'snippet_swd',
    'T50 Quote': 't50_swd',
    'T50 Quote-Free Context Length Matched': 't50freelength_swd',
    'Non-Literary Baseline': 'nonlit_swd'
}
swd_score_cols = list(swd_column_map.values())

try:
    PROJECT_ROOT = find_project_root()
    # This path points to the cache file created by generate_swd_stats.py
    swd_metrics_path = PROJECT_ROOT / 'Analysis' / 'Outputs' / 'stepwise_distance' / 'swd_scores_cache.csv'
    print(f"Project root found: {PROJECT_ROOT}")
    print(f"Constructed absolute path to SWD metrics: {swd_metrics_path}")

    # --- 2. Load SWD Data ---
    swd_df = pd.read_csv(swd_metrics_path)

    # --- 3. Select Score Columns and Merge ---
    # The cache file contains both text and score columns. We only want the scores.
    # The row order is preserved from the original analysis script, so we can merge on the index.
    df = df.merge(
        swd_df[swd_score_cols],
        left_index=True,
        right_index=True,
        how='left'
    )

    print("✓ Stepwise Distance metrics successfully merged into the main DataFrame.")

    # --- 4. Display Head of New Columns ---
    print("\nHead of the newly added Stepwise Distance columns:")
    print(df[swd_score_cols].head())

except FileNotFoundError:
    print(f"ERROR: SWD metrics cache not found at '{swd_metrics_path}'.")
    print("Please run 'Analysis/generate_swd_stats.py' first to generate the cache file.")
except KeyError:
    print("ERROR: One or more expected SWD columns were not found in the cache file.")
    print(f"Expected columns: {swd_score_cols}")
    print(f"Found columns: {list(swd_df.columns)}")
except Exception as e:
    print(f"An error occurred during Stepwise Distance integration: {e}")

In [ ]:
# Save the master dataframe with all metrics
manager.save_master_df(df)